In [ ]:
from src.data_prep.train_test_split import TrainTestSplit, prepare_development_and_holdout
from src.evaluation.cross_validation import stratified_k_fold
from src.evaluation.evaluator import ModelEvaluator
from src.evaluation.model_comparison import ModelComparison, format_comparison_table
from src.data_prep.preprocessing import preproc_MSCOHE
from src.evaluation.threshold_tuning import ThresholdTuner
from src.evaluation.holdout import HoldoutEvaluator
from src.data_prep.feature_selection import select_features
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


In [ ]:
# notebooks/model_selection.ipynb

# 1. data

splitter = TrainTestSplit(train_size=0.6, val_size=0.2, test_size=0.2, seed=42)
parts = splitter.split_csv()          
development, holdout = prepare_development_and_holdout(parts)
X_dev, y_dev = development["X"], development["y"]
X_hold, y_hold = holdout["X"], holdout["y"]
print(X_dev.shape, y_dev.shape, X_hold.shape, y_hold.shape)


In [ ]:

# 2. shared folds

folds = stratified_k_fold(y_dev, k=5, seed=42)
print(len(folds), "folds")


In [ ]:
#2.5a input prep 

#imagine for a second that there is a feature_selection.py and a function select_features(X) -> (X_subset, num_mask, cat_mask, kept_names)
X_dev_sub, NUM_MASK, CAT_MASK, KEPT_NAMES = select_features(
    X_dev, feature_names=feature_names,   # feature_names depuis l'en-tête CSV
)
print(f"{X_dev_sub.shape[1]} features retenues "
      f"({NUM_MASK.sum()} num, {CAT_MASK.sum()} cat)")
      


In [ ]:
#2.5b input prep 
assert (NUM_MASK & CAT_MASK).sum() == 0
assert (NUM_MASK | CAT_MASK).sum() == X_dev.shape[1]

factory = lambda: preproc_MSCOHE(num_mask=NUM_MASK, cat_mask=CAT_MASK)

In [ ]:

# 3. baseline comparison


candidates = {
    "Baseline":             ModelEvaluator("always_negative_baseline"),
    "Least squares":        ModelEvaluator("least_squares"),
    "MSE (GD)":             ModelEvaluator("mean_squared_error_gd", max_it=1000, gamma=0.01),
    "MSE (SGD)":            ModelEvaluator("mean_squared_error_sgd", max_it=1000, gamma=0.01, seed=42),
    "Ridge":                ModelEvaluator("ridge_regression", lmbd=0.1),
    "Logistic":             ModelEvaluator("logistic_regression", max_it=1000, gamma=0.01),
    "Regularized logistic": ModelEvaluator("reg_logistic_regression", lmbd=0.1, max_it=1000, gamma=0.01),
    "Linear SVM":           ModelEvaluator("svm", lmbd=0.1, max_it=1000, gamma=0.01),
}
comparison = ModelComparison(candidates, folds, fold_seed=42,
                             preprocessor_factory=factory)
results = comparison.run(X_dev, y_dev)
print(format_comparison_table(results))


In [ ]:

# 4. Tuning hyperparams 
# choose 2/3 models and select of few configs for them 
configurations = {
    "ridge_regression": [{"lmbd": l} for l in (1e-4, 1e-3, 1e-2, 1e-1, 1.0)],
    "reg_logistic_regression": [
        {"lmbd": 1e-3, "max_it": 1000, "gamma": 0.01},
        {"lmbd": 1e-2, "max_it": 1000, "gamma": 0.01},
        {"lmbd": 1e-3, "max_it": 5000, "gamma": 0.05},
    ],
}
comparison2 = ModelComparison.from_configurations(
    configurations, folds, fold_seed=42, preprocessor_factory=factory,
)
tuned = comparison2.run(X_dev, y_dev)
print(format_comparison_table(tuned))


In [ ]:

# 5. Tuning seuil

tuner = ThresholdTuner(
    {
        "ridge_regression":        [-0.5, -0.25, 0.0, 0.25, 0.5],
        "reg_logistic_regression": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
    },
    folds,
)
tuned_thr = tuner.run(tuned, y_dev)
print(format_comparison_table(tuned_thr))


In [ ]:

# 6. Report + plot
Path("results").mkdir(exist_ok=True)
comparison2.save_report(
    tuned_thr, "results/model_comparison.json",
    metadata={"dataset": "BRFSS", "preprocessing": "preproc_MSCOHE",
              "num_mask": NUM_MASK.tolist(), "cat_mask": CAT_MASK.tolist()},
)
fig, ax = plot_model_comparison(tuned_thr)
fig.savefig("results/model_comparison_f1.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:

# 7. Holdout final

best_name = next(iter(tuned_thr))     # sorted by mean F1
best_config = tuned_thr[best_name]["configuration"]
final = HoldoutEvaluator(best_config, preprocessor_factory=factory)
holdout_result = final.evaluate(development, holdout)
final.save_report(
    holdout_result, "results/holdout_evaluation.json",
    metadata={"selected_candidate": best_name,
              "selection_data": "development_oof",
              "preprocessing": "preproc_MSCOHE"},
)
print(holdout_result["metrics"])
print(holdout_result["confusion_matrix"])